# 07: Quad Trees

*Authors: Felix Espey, Kevin Buchin*

This notebook serves as supplementary learning material for the course **Geometric Algorithms**.
It showcases and explains implementations of algorithms presented in the corresponding lecture, and elaborates on some practical considerations concerning their use.
Furthermore, it offers interactive visualisations and animations.

## Table of Contents

1. Introduction
2. Quad Tree implementation and Construction

## Introduction

This notebook revisits the lecture(s) about

In [1]:
from typing import Optional

from modules import Point, Rectangle, QuadTreeAnimator, PointSetInstance, VisualisationTool, AnimationObject, QuadTreeMode

class Quadtree:
    def __init__(self):
        self.NE : Optional[Quadtree] = None
        self.NW : Optional[Quadtree] = None
        self.SE : Optional[Quadtree] = None
        self.SW : Optional[Quadtree] = None
        self.PARENT : Optional[Quadtree] = None
        self.points : list[Point] = []
        self.area : Optional[Rectangle] = None

    @property
    def isDummy(self) -> bool:
        return self.area is None

    @property
    def isLeaf(self) -> bool:
        return (self.NE is None and
                self.NW is None and
                self.SE is None and
                self.SW is None)

def construct_quad_tree(points : list[Point], animator : QuadTreeAnimator) -> Quadtree:
    def construct_recursive(cur_points : list[Point], area : Rectangle) -> Quadtree:
        node = Quadtree()
        node.area = area
        animator.set_node(cur_points, area)
        node.points = cur_points
        if len(cur_points) <= 1:
            return node
        # subareas:
        area_s, area_n = area.split_y(area.lower + (area.upper - area.lower) / 2)
        area_nw, area_ne = area_n.split_x(area.left + (area.right - area.left) / 2)
        area_sw, area_se = area_s.split_x(area.left + (area.right - area.left) / 2)
        # subsets of points:
        points_nw = []
        points_ne = []
        points_sw = []
        points_se = []
        for cur_point in cur_points:
            if cur_point.y <= area_s.upper:
                # point in lower half
                if cur_point.x <= area_sw.right:
                    points_sw.append(cur_point)
                else:
                    points_se.append(cur_point)
            else:
                if cur_point.x <= area_nw.right:
                    points_nw.append(cur_point)
                else:
                    points_ne.append(cur_point)
        animator.nw()
        node.NW = construct_recursive(points_nw, area_nw)
        animator.parent()
        animator.ne()
        node.NE = construct_recursive(points_ne, area_ne)
        animator.parent()
        animator.sw()
        node.SW = construct_recursive(points_sw, area_sw)
        animator.parent()
        animator.se()
        node.SE = construct_recursive(points_se, area_se)
        animator.parent()
        node.NW.parent = node
        node.NE.parent = node
        node.SW.parent = node
        node.SE.parent = node
        return node
    return construct_recursive(points, Rectangle(Point(0,0), Point(400,400)))


def alg_construct_quadtree(points : set[Point]) -> AnimationObject:
    animator = QuadTreeAnimator()
    root = construct_quad_tree(list(points), animator)
    return animator

In [2]:
instance = PointSetInstance()
instance._default_number_of_random_points = 15
instance._random_points_mode = 1
vis = VisualisationTool(400,400, instance)
vis.register_algorithm("Construct Quadtree" ,alg_construct_quadtree, QuadTreeMode())
vis.display()
vis._animation_checkbox.value = True
vis._random_button.click()
vis._algorithm_buttons[0].click()

[]
[<Direction.NW: 0>]
[<Direction.NW: 0>, <Direction.NW: 0>]
[<Direction.NW: 0>, <Direction.NE: 1>]
[<Direction.NW: 0>, <Direction.SW: 2>]
[<Direction.NW: 0>, <Direction.SE: 3>]
[<Direction.NE: 1>]
[<Direction.NE: 1>, <Direction.NW: 0>]
[<Direction.NE: 1>, <Direction.NE: 1>]
[<Direction.NE: 1>, <Direction.SW: 2>]
[<Direction.NE: 1>, <Direction.SE: 3>]
[<Direction.SW: 2>]
[<Direction.SW: 2>, <Direction.NW: 0>]
[<Direction.SW: 2>, <Direction.NW: 0>, <Direction.NW: 0>]
[<Direction.SW: 2>, <Direction.NW: 0>, <Direction.NE: 1>]
[<Direction.SW: 2>, <Direction.NW: 0>, <Direction.NE: 1>, <Direction.NW: 0>]
[<Direction.SW: 2>, <Direction.NW: 0>, <Direction.NE: 1>, <Direction.NE: 1>]
[<Direction.SW: 2>, <Direction.NW: 0>, <Direction.NE: 1>, <Direction.SW: 2>]
[<Direction.SW: 2>, <Direction.NW: 0>, <Direction.NE: 1>, <Direction.SE: 3>]
[<Direction.SW: 2>, <Direction.NW: 0>, <Direction.SW: 2>]
[<Direction.SW: 2>, <Direction.NW: 0>, <Direction.SE: 3>]
[<Direction.SW: 2>, <Direction.NE: 1>]
[<Direc